# Notebook 12 – Feature Relationships

Topics: Correlation Matrix, Multicollinearity, Variance Inflation Factor (VIF), Feature Importance, Feature Selection Basics


# 1. Correlation Matrix

### Definition

A correlation matrix shows pairwise correlation values between multiple numerical features.

### Real-life Example

In a house dataset, it can show relationships among Area, Bedrooms, Age, and Price.

### Formula / Numerical Example

Pearson correlation:
$$r=\frac{\sum (x_i-\bar{x})(y_i-\bar{y})}{\sqrt{\sum(x_i-\bar{x})^2\sum(y_i-\bar{y})^2}}$$
Where $r$ = correlation coefficient, $x_i,y_i$ = individual values, and $\bar{x},\bar{y}$ = sample means. Values range from -1 to +1.

### Python Implementation


In [1]:
import pandas as pd
df=pd.DataFrame({
    "Area":[1000,1200,1500,1800,2000],
    "Bedrooms":[2,2,3,3,4],
    "Age":[20,18,15,10,5],
    "Price":[40,45,60,75,90]
})
print(df.corr(numeric_only=True))

              Area  Bedrooms       Age     Price
Area      1.000000  0.942128 -0.982871  0.991292
Bedrooms  0.942128  1.000000 -0.949158  0.962660
Age      -0.982871 -0.949158  1.000000 -0.995963
Price     0.991292  0.962660 -0.995963  1.000000


### AI/ML Example

A correlation matrix helps identify strong linear relationships and possible redundant numerical features before model training.


# 2. Multicollinearity

### Definition

Multicollinearity occurs when predictor features in a regression model are strongly linearly related to each other.

### Real-life Example

Area in square feet and the same Area in square meters contain nearly the same information.

### Formula / Numerical Example

A common diagnostic is:
$$VIF_j=\frac{1}{1-R_j^2}$$
Where $VIF_j$ = VIF for predictor $j$ and $R_j^2$ = R-squared from predicting that feature using the other predictors.
If $R_j^2=0.90$, then $VIF=10$.

### Python Implementation


In [3]:
import pandas as pd
df=pd.DataFrame({
    "Area_sqft":[1000,1200,1500,1800,2000],
    "Area_sqm":[92.9,111.5,139.4,167.2,185.8],
    "Bedrooms":[2,2,3,3,4]
})
print(df.corr(numeric_only=True))

           Area_sqft  Area_sqm  Bedrooms
Area_sqft   1.000000  1.000000  0.942128
Area_sqm    1.000000  1.000000  0.942229
Bedrooms    0.942128  0.942229  1.000000


### AI/ML Example

Multicollinearity can make regression coefficients unstable and harder to interpret. Redundant features may be removed or combined depending on the problem.


# 3. Variance Inflation Factor (VIF)

### Definition

VIF measures how strongly a predictor is linearly explained by the other predictor features.

### Real-life Example

Age and years of experience may contain overlapping information, which VIF can help detect.

### Formula / Numerical Example

$$VIF_j=\frac{1}{1-R_j^2}$$
Where $VIF_j$ = Variance Inflation Factor and $R_j^2$ = R-squared for predictor $j$ against the remaining predictors.
VIF = 1 means no linear explanation by the other predictors. Larger values indicate more multicollinearity.

### Python Implementation


In [5]:
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor

X=pd.DataFrame({
    "Area":[1000,1200,1500,1800,2000,2200],
    "Bedrooms":[2,2,3,3,4,4],
    "Age":[20,18,15,10,5,3]
})

vif=pd.DataFrame()
vif["Feature"]=X.columns
vif["VIF"]=[variance_inflation_factor(X.values,i)
            for i in range(X.shape[1])]
print(vif)

    Feature         VIF
0      Area  179.716655
1  Bedrooms  178.471419
2       Age    2.156961


### AI/ML Example

VIF is useful when checking multicollinearity in regression predictors. High values should be investigated together with domain knowledge rather than used as an automatic removal rule.


# 4. Feature Importance

### Definition

Feature importance estimates how much features contribute to a model's predictions according to a particular model and importance method.

### Real-life Example

In house-price prediction, Area may have greater model importance than another feature.

### Formula / Numerical Example

There is no single universal feature-importance formula.

For example, normalized model importances might be:
$$Area=0.60,\quad Bedrooms=0.25,\quad Age=0.15$$
and their total is 1. Larger values mean greater importance according to that model and method.

### Python Implementation


In [7]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

X=pd.DataFrame({
    "Area":[1000,1200,1500,1800,2000,2200],
    "Bedrooms":[2,2,3,3,4,4],
    "Age":[20,18,15,10,5,3]
})
y=[40,45,60,75,90,100]

model=RandomForestRegressor(n_estimators=100,random_state=42)
model.fit(X,y)

importance=pd.Series(model.feature_importances_,index=X.columns)
print(importance.sort_values(ascending=False))

Area        0.401409
Bedrooms    0.312532
Age         0.286059
dtype: float64


### AI/ML Example

Feature importance helps understand which inputs a trained model relies on most. Importance does not mean causation and can vary by model and method.


# 5. Feature Selection Basics

### Definition

Feature selection means choosing useful input features while excluding irrelevant, redundant, or unnecessarily noisy features.

### Real-life Example

A dataset may contain 50 columns, while only a smaller subset provides useful predictive information.

### Formula / Numerical Example

Three common approaches are:
1. Filter Methods
2. Wrapper Methods
3. Embedded Methods

A simple correlation filter may use:
$$|r|\geq t$$
Where $r$ = feature-target correlation and $t$ = chosen threshold. The threshold depends on the problem.

### Python Implementation


In [9]:
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_regression

X=pd.DataFrame({
    "Area":[1000,1200,1500,1800,2000,2200],
    "Bedrooms":[2,2,3,3,4,4],
    "Age":[20,18,15,10,5,3]
})
y=[40,45,60,75,90,100]

selector=SelectKBest(score_func=f_regression,k=2)
selector.fit(X,y)

print("Selected Features:",list(X.columns[selector.get_support()]))

Selected Features: ['Area', 'Age']


### AI/ML Example

Feature selection can simplify models, reduce unnecessary inputs, and sometimes improve generalization. It should be fitted using training data only to avoid data leakage.
